# Approach comparison for Photo AI Platform

Goal: compare simpler alternatives to a heavy caption-first clustering pipeline.

The existing notebooks are kept untouched. This notebook uses the lightweight benchmark in `ml/experiments/photo_strategy_benchmark.py` so the experiment can be rerun without GPU, Qdrant or model downloads.

## Hypothesis

For production, the core path should be:

1. metadata filters,
2. image embeddings in a vector database,
3. optional graph links for grouping and recommendations,
4. captions only as offline enrichment.

This should be easier to scale and maintain than generating captions for every image and tuning text clusters.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'ml' / 'experiments'))

from photo_strategy_benchmark import (
    generate_photos,
    metadata_facets,
    text_caption_clustering,
    embedding_knn_graph,
    hybrid_similarity_graph,
    timed,
)

photos = generate_photos(count=800, seed=42)
len(photos)

In [ ]:
rows = [
    timed('metadata_facets', metadata_facets, photos),
    timed('text_caption_clustering', text_caption_clustering, photos),
    timed('embedding_knn_graph', embedding_knn_graph, photos),
    timed('hybrid_similarity_graph', hybrid_similarity_graph, photos),
]

for row in rows:
    print(row)

## Reading the result

- `metadata_facets` should be almost free and becomes the baseline filter layer.
- `text_caption_clustering` is sensitive to caption quality and can require tuning.
- `embedding_knn_graph` is slow here because it is exact pure Python; in production, Qdrant replaces this part.
- `hybrid_similarity_graph` is the most product-friendly grouping layer because it combines embeddings, shooting, time and people signals.

## Recommended next implementation

1. Keep `libraries`, `shootings`, `photos` and metadata in PostgreSQL.
2. Store images in object storage.
3. Compute one image embedding per photo.
4. Store vectors in Qdrant.
5. Build graph edges asynchronously from nearest neighbors and metadata.
6. Add captions only later for richer natural-language queries.